In [31]:
import requests
import os
import pandas as pd
from tqdm import tqdm
from bs4 import BeautifulSoup as bs
import csv

In [3]:
abbrevs_path = 'input/nfl_team_abbrevs.csv'
teams_path = 'input/teams/'
abbrevs = pd.read_csv(abbrevs_path)

### Tree-shaking empty team files

In [5]:
# Define the folder path containing the HTML files
folder_path = 'input/teams'

# Iterate through all files in the folder
for filename in os.listdir(folder_path):
    if filename.endswith('.htm'):
        file_path = os.path.join(folder_path, filename)

        # Read the content of the HTML file
        with open(file_path, 'r', encoding='utf-8') as file:
            content = file.read()

        # Parse the HTML content with BeautifulSoup
        soup = bs(content, 'html.parser')

        # Check if the content is None or if the page is empty (has no content)
        if soup is None or not soup.find(True):  # `find(True)` looks for any tag, returns None if none found
            print(f"Deleting empty file: {file_path}")
            os.remove(file_path)  # Delete the file


### Scraping Every Team

In [6]:
url = 'https://www.pro-football-reference.com/teams/buf/2024.htm'
soup = bs(requests.get(url).content,'html.parser')
soup.find_all('div')[11]

<div id="meta">
<div class="media-item logo loader">
<img alt="2024 Buffalo Bills Logo" class="teamlogo" src="https://cdn.ssref.net/req/202412261/tlogo/pfr/buf-2024.png"/>
<p><a href="http://www.sportslogos.net/">via Sports Logos.net</a></p>
<p><a href="https://www.sports-reference.com/blog/2016/06/redesign-team-and-league-logos-courtesy-sportslogos-net/">About logos</a></p>
</div>
<div data-template="Partials/Teams/Summary">
<h1>
<span>2024</span>
<span>Buffalo Bills</span>
<span class="header_end">Rosters, Stats, Schedule, Team Draftees, Injury Reports</span>
</h1>
<div class="prevnext">
<a class="button2 prev" href="/teams/buf/2023.htm">Previous Season</a>
</div>
<p>
<strong>Record:</strong> 13-4-0, 1st in
	<a href="/years/2024">AFC East Division</a>
   (<a href="/teams/buf/2024_games.htm">Schedule and Results</a>)
</p>
<p><strong>Coach:</strong>
<a href="/coaches/McDeSe0.htm">Sean McDermott</a> (13-4-0)</p>
<p><strong>Points For:</strong> 525 (30.9/g) 2nd of 32</p>
<p><strong>Point

In [44]:
for team in abbrevs['Abbrev']:
    team = team.lower()
    base = f'https://www.pro-football-reference.com/teams/{team}'
    for year in range(2024,2025):
        link = f'{base}/{year}.htm'
        path = teams_path + link.split('/')[-2] + link.split('/')[-1]
        if not os.path.exists(path):
            # print(bs(requests.get(link).content,'html.parser').find(id='meta'))
            soup = bs(requests.get(link).content,'html.parser')
            html = soup.find_all('div')[11]
            with open(path, 'w') as f:
                text = f.write(str(html))

### 2024 Coaches


In [9]:
url = 'https://www.pro-football-reference.com/coaches/'
soup = bs(requests.get(url).content)

In [10]:
active = []
activeRows = []
for tr in soup.find_all('tr'):
    if tr.find('td', class_='right', attrs={'data-stat': 'year_max'}, string='2024'):
        activeRows.append(tr)
for row in activeRows:
    active.append(row.find('a')['href'])

In [38]:
BASE = 'https://www.pro-football-reference.com'
coaches = [coach.split('/')[-1].split('.')[0] for coach in active]

In [12]:
def clean_columns(df):
    df.columns = df.columns.get_level_values(1)
    dupIndex = list(df.columns).index('W-L%',list(df.columns).index('W-L%')+1)
    df.columns.values[dupIndex] = 'W-L% plyf'
    return df

In [13]:
# TODO: account for active coach or not
def get_coach(coach,folder='tables',fetch=False):
    url = f'https://www.pro-football-reference.com/coaches/{coach}.htm'
    file_name = f"input/coaches/{folder}/{url.split('/')[-1].split('.')[0]}.csv"

    if os.path.exists(file_name):
        return pd.read_csv(file_name,index_col=0)

    else :
        ret = requests.get(url).text
        ret = pd.read_html(ret)[0]
        ret = clean_columns(ret)
        ret.to_csv(file_name)
        return ret

In [40]:
pd.DataFrame(coaches,columns=['name']).to_csv('input/coaches24.csv')

In [21]:
def clean_rows(coach,footerBool=False):
    df = get_coach(coach)
    footer = df[~(df['Year'].str.isdigit())]
    rows = df[df['Year'].str.isdigit()].copy()

    if 'Num' not in df.columns :
        rows.insert(len(df.columns)-1,'Num',0)
        rows.insert(len(df.columns)-1,'Won',0)
    
    rows = rows.fillna(0)
    
    for col in ['Age','Year']:
        rows[col] = rows[col].astype(int)
    
    rows['Notes'] = rows['Notes'].astype(str)

    return rows if not footerBool else footer

In [35]:
def clean_coach(coachName,folder='2024',save=False):
    coach = get_coach(coachName,folder=folder)
    coach["int"] = pd.to_numeric(coach["Year"], errors="coerce").notna()
    ret = coach[coach['int']].copy()
    max_year = ret['Year'].max()
    max_team = ret.loc[ret['Year'] == max_year]['Tm'].values[0]

    coach['Keep'] = True
    for i in range(len(coach)-1, -1, -1):
        row = coach.iloc[i]
        if row.int and row.Tm == max_team:
            coach.at[i,'Keep'] = False
        elif row.int:
            break
    coach = coach[coach['Keep']]
    coach = coach.drop(columns=['int','Keep'])
    if save : 
        coach.to_csv(f'input/coaches/tables/{coachName}.csv')
        print(f'Saved: {coachName}.csv')
    return coach

In [32]:
# # Deleting files from tables of 2024 coaches
# # Paths
# csv_file_path = 'input/coaches24.csv'  # Replace with your CSV file path
# folder_path = 'input/coaches/tables'  # Replace with the folder containing files to delete

# # Read the CSV and get the list of file names
# with open(csv_file_path, 'r') as csv_file:
#     csv_reader = csv.reader(csv_file)
#     file_names = [row[1]+'.csv' for row in csv_reader]  # Assuming file names are in the first column

# # Delete the files
# for file_name in file_names:
#     file_path = os.path.join(folder_path, file_name)
#     if os.path.exists(file_path):
#         os.remove(file_path)
#         print(f"Deleted: {file_path}")
#     else:
#         print(f"File not found: {file_path}")


In [37]:
for coach in tqdm(coaches):
    clean_coach(coach,save=True)

100%|██████████| 35/35 [00:00<00:00, 400.53it/s]

Saved: ReidAn0.csv
Saved: TomlMi0.csv
Saved: McCaMi0.csv
Saved: HarbJo0.csv
Saved: PaytSe0.csv
Saved: McDeSe0.csv
Saved: McVaSe0.csv
Saved: ShanKy0.csv
Saved: LaFlMa0.csv
Saved: PedeDo0.csv
Saved: HarbJi0.csv
Saved: QuinDa0.csv
Saved: BowlTo0.csv
Saved: SiriNi0.csv
Saved: TaylZa0.csv
Saved: CampDa1.csv
Saved: StefKe0.csv
Saved: OConKe0.csv
Saved: MorrRa0.csv
Saved: McDaMi0.csv
Saved: AlleDe0.csv
Saved: RyanDe0.csv
Saved: SaleRo0.csv
Saved: DaboBr0.csv
Saved: SteiSh0.csv
Saved: EberMa0.csv
Saved: GannJo0.csv
Saved: MacdMi0.csv
Saved: PierAn0.csv
Saved: CanaDa0.csv
Saved: MayoJe0.csv
Saved: CallBr0.csv
Saved: RizzDa0.csv
Saved: UlbrJe0.csv
Saved: BrowTh0.csv


### Getting Coach Names

In [47]:
url = 'https://www.pro-football-reference.com/coaches/'
df = pd.read_html(url)[0]

In [71]:
df.tail()

,Rk,Coach,Yrs,From,To,G,W,L,T,W-L%,...,Yr plyf,G plyf,W plyf,L plyf,W-L%.1,AvRk,BstRk,Chmp,SBwl,Conf
526,527,Johnny Murphy,1,1924,1924,4,0,4,0,0.0,...,0.0,0.0,0.0,0.0,NaN,16.0,16,0.0,0.0,0.0
527,528,Al Nesser,1,1926,1926,2,0,1,1,0.0,...,0.0,0.0,0.0,0.0,NaN,16.0,16,0.0,0.0,0.0
528,529,Tam Rose,1,1921,1921,1,0,1,0,0.0,...,0.0,0.0,0.0,0.0,NaN,18.0,18,0.0,0.0,0.0
529,530,Lenny Sachs,1,1926,1926,4,0,4,0,0.0,...,0.0,0.0,0.0,0.0,NaN,21.0,21,0.0,0.0,0.0
530,531,Giff Smith,1,2023,2023,3,0,3,0,0.0,...,NaN,NaN,NaN,NaN,NaN,4.0,4,NaN,NaN,NaN


In [49]:
soup = bs(requests.get(url).content)

In [67]:
coaches = []
seen = set()
for tag in soup.find_all('a'):
    coach = tag['href'].split('/')[-1].split('.')[0]
    if '/coaches/' in tag['href'] and coach not in seen:
        coaches.append(coach)
        seen.add(coach)

In [76]:
"jklhsg"[:-1]

'jklhs'

In [77]:
def clean_hof(name):
    if name[-1] == '+' : return name[:-1]
    return name

In [83]:
key = pd.DataFrame(coaches,columns=['id']).iloc[:-1,:]
key['name'] = df['Coach'].apply(clean_hof)
key['to'] = df['To']
key.to_csv('input/coaches.csv')